# Phase 6: Model Save/Load & Inference

## What you'll learn
- Save/load model weights (state_dict)
- Full checkpoints (model + optimizer + epoch)
- `model.eval()` vs `model.train()`
- Inference with `torch.no_grad()`
- Export to ONNX
- TorchScript (trace & script)

---

In [ ]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))

model = SimpleModel()
print(model)

## 6.1 Save/Load State Dict (Recommended)

The `state_dict` contains all learnable parameters. This is the **preferred** way to save models.

In [ ]:
# Save weights only
torch.save(model.state_dict(), 'model_weights.pth')
print("Saved state_dict")

# Load weights into a new model instance
loaded_model = SimpleModel()  # must create same architecture first
loaded_model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
loaded_model.eval()
print("Loaded state_dict")

# Verify weights match
for (n1, p1), (n2, p2) in zip(model.named_parameters(), loaded_model.named_parameters()):
    print(f"{n1}: match={torch.equal(p1, p2)}")

## 6.2 Full Checkpoint (Resume Training)

Save everything needed to resume training: model, optimizer, epoch, loss, etc.

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

# Save checkpoint
checkpoint = {
    'epoch': 10,
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'best_val_loss': 0.05,
}
torch.save(checkpoint, 'checkpoint.pth')
print("Checkpoint saved")

# Load checkpoint
ckpt = torch.load('checkpoint.pth', weights_only=False)

resume_model = SimpleModel()
resume_optimizer = torch.optim.AdamW(resume_model.parameters(), lr=1e-3)

resume_model.load_state_dict(ckpt['model_state_dict'])
resume_optimizer.load_state_dict(ckpt['optimizer_state_dict'])
start_epoch = ckpt['epoch']

print(f"Resumed from epoch {start_epoch}")
print(f"Best val loss: {ckpt['best_val_loss']}")

## 6.3 train() vs eval() Modes

| Mode | Dropout | BatchNorm | Use When |
|------|---------|-----------|----------|
| `model.train()` | Active (random) | Uses batch stats | Training |
| `model.eval()` | Disabled | Uses running stats | Validation/Inference |

In [ ]:
model_with_dropout = nn.Sequential(
    nn.Linear(10, 5),
    nn.Dropout(0.5),
    nn.Linear(5, 1)
)

x = torch.ones(1, 10)

# Train mode — dropout active, outputs vary
model_with_dropout.train()
print("Train mode outputs (vary due to dropout):")
for i in range(3):
    print(f"  Run {i+1}: {model_with_dropout(x).item():.4f}")

# Eval mode — dropout disabled, outputs consistent
model_with_dropout.eval()
print("\nEval mode outputs (consistent):")
for i in range(3):
    print(f"  Run {i+1}: {model_with_dropout(x).item():.4f}")

## 6.4 Inference with torch.no_grad()

Always use `torch.no_grad()` during inference — saves memory and speeds up computation.

In [ ]:
model = SimpleModel()
model.eval()

dummy_input = torch.randn(1, 784)

# Correct inference pattern
with torch.no_grad():
    logits = model(dummy_input)
    probs = torch.softmax(logits, dim=1)
    predicted_class = probs.argmax(dim=1)

print(f"Logits: {logits}")
print(f"Probabilities: {probs}")
print(f"Predicted class: {predicted_class.item()}")
print(f"Confidence: {probs.max().item():.2%}")

## 6.5 Export to ONNX

ONNX (Open Neural Network Exchange) lets you run models in other frameworks (TensorRT, ONNX Runtime, etc.).

In [ ]:
model = SimpleModel()
model.eval()

dummy = torch.randn(1, 784)

torch.onnx.export(
    model,
    dummy,
    'model.onnx',
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch'}, 'output': {0: 'batch'}}
)
print("Exported to ONNX")

## 6.6 TorchScript — Optimized Serialization

TorchScript converts your model to a format that can run **without Python** (C++ runtime).

Two methods:
- `torch.jit.trace` — records operations with example input (no control flow)
- `torch.jit.script` — analyzes Python code (supports if/for)

In [ ]:
model = SimpleModel()
model.eval()
dummy = torch.randn(1, 784)

# Trace
traced = torch.jit.trace(model, dummy)
traced.save('model_traced.pt')
print(f"Traced output: {traced(dummy).shape}")

# Script (better for models with control flow)
scripted = torch.jit.script(model)
scripted.save('model_scripted.pt')
print(f"Scripted output: {scripted(dummy).shape}")

# Load without needing the original class definition
loaded = torch.jit.load('model_traced.pt')
print(f"Loaded output: {loaded(dummy).shape}")

## ✅ Phase 6 Summary

You now know how to:
- Save/load weights with `state_dict` (recommended)
- Save full checkpoints for resuming training
- Switch between `train()` and `eval()` modes
- Run inference efficiently with `torch.no_grad()`
- Export models to ONNX and TorchScript

**Next → Phase 7: CNNs (Computer Vision)**